### Метрики эффективности моделей машинного обучения

#### Цель работы

Научиться измерять эффективность моделей машинного обучения с помощью метрик, вибирать метрики исходя из задачи, разбивать датасет на обучающую и тестовую подвыборки.

#### Содержание работы

1. Загрузите данные о вероятности развития сердечного приступа, прилагающийся к этой работе (heart.csv).
1. Обучите на этих данных простую модель логистической регрессии и выведите метрику точности (accuracy).
1. Разделите датасет на две части - первую половину используйте для обучения, а вторую - для оценки точности. Сравните значения метрик.
1. Разделите датасет на две части случайным образом. Повторите анализ.
1. Разделите датасет с помощью библиотечной функции. Повторите анализ несколько раз.
1. Постройте матрицу классификации и отчет о классификации для обученной модели для обучающей и тестовой выборок. Проинтерпретируйте полученные значения.
1. Подсчитайте для построенной модели значение всех метрик эффективности классификации на тестовой и обучающей выборках. Нужно использовать следующие метрики: accuracy, precision, recall, f1.

#### Методические указания

In [128]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

После чтения файла вы должны увидеть примерно такую таблицу при выводе первых строк датасета:

In [129]:
data = pd.read_csv('data/heart.csv')

print(data.head())

print(data.info())
print(data.describe())

   age  sex  cp  trtbps  chol  fbs  restecg  thalachh  exng  oldpeak  slp  \
0   63    1   3     145   233    1        0       150     0      2.3    0   
1   37    1   2     130   250    0        1       187     0      3.5    0   
2   41    0   1     130   204    0        0       172     0      1.4    2   
3   56    1   1     120   236    0        1       178     0      0.8    2   
4   57    0   0     120   354    0        1       163     1      0.6    2   

   caa  thall  output  
0    0      1       1  
1    0      2       1  
2    0      2       1  
3    0      2       1  
4    0      2       1  
<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trtbps    303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-nul

Как всегда выделим целевую переменную, сразу же импортируем и обучим модель логистической регрессии:

In [130]:
y = data["output"]
x = data.drop("output", axis=1)

from sklearn.linear_model import LogisticRegression

logistic = LogisticRegression().fit(x, y)
logistic.score(x, y)

/home/kai/ML_course/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8547854785478548

Данная модель показывает более 85% точности. Но эта оценка качества модели является завышенной (оптимистичной), так как она оценена именно по тем данным, на которых модель училась. Чтобы оценить именно обобщающую способность модели, то есть ее способность правильно предсказывать значение целевой переменной для тех объектов, которые она не учитывала при подстройке параметров, необходимо оценить модель на другой выборке. Для этого нужно разделить исходный датасет на две части. Обучающая выборка (первая часть) будет использоваться для нахождения оптимальных значений внутренних параметров модели, а тестовая выборка (вторая часть) - для оценки качества полученной модели.

##### Разделение выборки

Делить датасет можно разными способами. Важно, чтобы каждый конкретный объект попал только в одну выборку - либо тестовую, либо обучающую. То есть эти части должны быть непересекающиеся. Самый простой способ - просто взять какое-то количество объектов в начале датафрейма в обучающую выборку, а остальные - в тестовую. То есть мы просто берем несколько первых строчек (например, 200) для обучающей выборки:

In [131]:
x_train, y_train = x[:200], y[:200]

Важно убедиться, что все в порядке с формами получившихся массивов:

In [132]:
x_train.shape, y_train.shape

((200, 13), (200,))

Итак, первые 200 строк попали в обучающую выборку. Тогда тестовую выборку составять оставшиеся строки датасета:

In [133]:
x_test, y_test = x[200:], y[200:]
x_test.shape, y_test.shape

((103, 13), (103,))

Получилось, что в тестовой выборке осталось 103 объекта. Теперь можно заново обучить модель классификации и оценить ее качество. Обратите внимание, что мы вызываем метод fit() именно на обучающей части датасета. А вот эффективность измеряем сначала на обучающей части, а затем на тестовой, чтобы сравнить:

In [134]:
logistic_test = LogisticRegression().fit(x_train, y_train)
logistic_test.score(x_train, y_train), logistic_test.score(x_test, y_test)

/home/kai/ML_course/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(0.9, 0.5436893203883495)

Получилось, что на обучающей выборки точность модели даже немного повысилась, до 90%. Это произошло за счет того, что в обучающей выборке меньше данных, чем в целом датасете. А чем меньше точек, тем проще модели под них подстроиться. А вот эффективность модели на тестовых данных стала сильно ниже - всего 54%. Это значит, что наша модель чуть лучше, чем простое угадывание. Хотя, как увидим дальше, на самом деле не все так плохо.

Вместо того, чтобы вручную задавать количество объектов, которые отводятся на обучающую выборку, можно выразить это количество через процент от всего объема датасета. Это даст нам универсальность:

В данном примере мы взяли 80% объектов для обучающей выборки и 20% - оставили на тестовую. Это довольно стандартная схема разбиения и мы в общем случае будем придерживаться именно ее. Вот какие формы получились у массивов:

In [135]:
N = int(x.shape[0] * 0.8)

x_train, y_train, x_test, y_test = x[:N], y[:N], x[N:], y[N:]
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((242, 13), (242,), (61, 13), (61,))

Давайте еще раз обучим и оценим модель, уже на новом разделении:

In [136]:
logistic_test = LogisticRegression().fit(x_train, y_train)
logistic_test.score(x_train, y_train), logistic_test.score(x_test, y_test)

/home/kai/ML_course/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(0.8884297520661157, 0.6229508196721312)

Мы видим, что точность на обучающей выборке (мы ее будем называть обучающей точностью) чуть опустилась - опять же это эффект количества точек - сейчас их чуть больше. Но и тестовая эффективность (точность модели, измеренная на тестовой выборке) тоже подросла - до 62%. Это потому, что чем больше примеров мы использовали для обучения, тем более качественной и обобщающей получается наша модель.

Но разница между обучающей и тестовой точностью у нашей модели все еще довольно высока. С чем это может быть связано? На самом деле, мы поступаем немного неправильно. Нам нужно более внимательно присмотреться к данным. Мы уже видели первые несколько строк датасета. Давайте еще выведем последние несколько строк:

In [137]:
data.tail()

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0


Можно заметить, что вы первой половине данных целевая переменная всегда положительная, а во второй - отрицательная. Другими словами, датасет отсортирован по значению целевой переменной. Поэтому наше разбиение имеет один неприятный эффект - в обучающей выборке скапливается большое количество положительных примеров, а в тестовой - только отрицательные. Другими словами, обучающая и тестовая выборки очень непохожи друг надруга. А для того, чтобы наше оценивание сработало, нужно, чтобы они были как можно более однородными.

Поэтому просто выбрать первые строки датасета обычно не очень рациональное решение. Даже если датасет не отсортирован по значению целевой переменной, он вполне может быть упорядочен по какому-то другому признаку, или хотя бы по времени добавления объектов в набор или базу данных. А такое упорядочение вносит систематическую ошибку выборки, которая смещает оценку качества модели.

Из-за этого делить данные на обучающую и тестовую выборки практически всегда нужно случайным образом. То есть выбрать случайное подмножество точек и поместить их в обучающий набор, а оставшиеся - в тестовый. Основная техническая трудность здесь состоит в том, что датасет у нас разделен на две переменные - матрицу признаков и вектор значений целевой переменной. Просто воспользоваться методом случайного выбора из массива мы не можем, так как когда мы повторим его два раза - для переменных x и y у нас будут выбраны не соответствующие друг другу объекты.

Для решения этой проблемы воспользуемся индексными масками. Заведем массив булевских значений, в которых количество истинных значений будет соответствовать желаемому объему обучающей выборки, а ложных - объему тестовой

In [138]:
mask = np.array([True] * N + [False] * (y.shape[0] - N))

В сумме должно получиться в точности количество объектов в полном датасете.Теперь перемешаем этот массив, используя стандартную функцию numpy:

In [139]:
from numpy.random import shuffle

shuffle(mask)
mask

array([ True,  True,  True, False, False,  True,  True, False,  True,
       False,  True,  True,  True, False,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True, False,  True,  True,  True,  True,
       False, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True, False,  True,  True,  True,  True,  True, False,
       False,  True,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True,  True, False,  True, False, False,  True,
        True,  True, False,  True,  True, False, False,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True, False,
        True, False,  True, False, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

Этот булев массив можно использовать как маску при индексировании исходного датасета. Если выбрать данные по этой маске, то в итог попадут только те элементы, которые стоят на тех местах, на которых в маске - истина:

In [140]:
x_train = x[mask]
x_train.shape

(242, 13)

Смысл этого приема в том, что мы можем применить одну и ту же маску к обоим частям массива. А если инвертировать эту маску, то выберем оставшиеся значения:

In [141]:
x_train, y_train, x_test, y_test = x[mask], y[mask], x[~mask], y[~mask]
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((242, 13), (242,), (61, 13), (61,))

После такого разделения опять обучим и оценим модель:

In [142]:
logistic_test = LogisticRegression().fit(x_train, y_train)
logistic_test.score(x_train, y_train), logistic_test.score(x_test, y_test)

/home/kai/ML_course/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(0.8512396694214877, 0.8032786885245902)

Мы видим, что теперь разница между тестовой и обучающей выборкой стала гораздо меньше - 88% и 77% соответственно. Это как раз обосновано более правильным разделением, которое дает однородные по своему составу части датасета.

Конечно, такая стандартная процедура как случайное разделение датасета на две части уже реализована в виде готовой функции - в реальной работе не придется каждый раз дублировать приведенный код. Такая функция лежит в пакете sklearn, который отвечает за методологию выбора модели - обэтом мы поговорим в следующих работах:

In [143]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((242, 13), (242,), (61, 13), (61,))

Обратите внимание, что библиотечная функция возвращает элементы в строго определенном порядке. Его придется либо запомнить, либо пользоваться готовыми сниппетами кода. 

##### Построение метрик качества классификации

До сих пор мы оценивали модель только по одной метрике эффективности. Но для более полного анализе этого недостаточно. В пакете _metrics_ собрано множество функций, которые позволяют исследовать поведение уже обученных моделей, в том числе - метрики эффективности. Импортируем несколько нужных нам инструментов:

In [144]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

Для использования большинства метрик необходимо передавать в эти функции два вектора - вектор истинных значений целевой переменной и вектор предсказанных значений. Истинные (эмпирические) значения - это часть исходного датасета. А предсказанные (теоретические) значения можно вычислить. Рассчитаем предсказанные значения отдельно для обучающей и для тестовой выборки:

In [145]:
y_test_pred = logistic_test.predict(x_test)
y_train_pred = logistic_test.predict(x_train)

Первым делом построим матрицу классификации
Она показывает, как часто и в каких классах модель ошибается. В идеальной модели ненулевые значения должны быть только на главной диагонали:

In [146]:
confusion_matrix(y_train, y_train_pred)

array([[ 87,  27],
       [ 14, 114]])

Обратите внимание, что для правильного анализа модели нужно строить все эти диагностические инструменты именно по тестовой выборке:

In [147]:
confusion_matrix(y_test, y_test_pred)

array([[19,  5],
       [ 2, 35]])

Именно эти данные покажут истинное поведение модели.

Еще один полезный инструмент - отчет о классификации:

In [148]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.90      0.79      0.84        24
           1       0.88      0.95      0.91        37

    accuracy                           0.89        61
   macro avg       0.89      0.87      0.88        61
weighted avg       0.89      0.89      0.88        61



Он показывает всю основную информацию по итогам классификации. Познакомьтесь со структурой данного отчета и проинтерпретируйте полученные результаты.

Что касается остальных метрик качества классификации, можно строить их отдельно:

In [149]:
precision_score(y_test, y_test_pred)

0.875

Но более удобно свести их все в таблицу при помощи датафрейма:

In [150]:
metrics = pd.DataFrame({
    "Train": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
    ],
    "Test": [
        accuracy_score(y_test, y_test_pred),
        precision_score(y_test, y_test_pred),
        recall_score(y_test, y_test_pred),
        f1_score(y_test, y_test_pred),
    ],
}, index = ["Accuracy", "Precision", "Recall", "F1"])

metrics

,Train,Test
Accuracy,0.830579,0.885246
Precision,0.808511,0.875000
Recall,0.890625,0.945946
F1,0.847584,0.909091


Здесь мы видим сравнение четырех разных основных метрик качества модели классификации по обучающей и по тестовой выборкам. Видно, что тестовые метрики все чуть ниже, чем обучающие. Это доволно типичное поведение моделей машинного обучения. Мы уже рассматривали причины этого. Но помните, что это не гарантируется и в конкретном случае, тестовые метрики могут быть как очень близки, так и даже выше. Все решает случай при рандомном разеделении выборки.

При оценке эффективности модели машинного обучения ориентирвоаться нужно только на тестовые метрики. Но сравнение уровня тестовой и обучающей метрики - это один их основных инструментов диагностики модели машинного обучения.

### Контрольные вопросы

**1. Почему для анализа модели нужно применять несколько метрик эффективности?**  
Одна метрика (например, accuracy) может быть неинформативна при несбалансированных классах: модель, предсказывающая всегда один класс, покажет высокую точность, но будет бесполезна. Precision, recall, F1 и другие метрики позволяют увидеть ошибки по каждому классу отдельно, оценить баланс между ложноположительными и ложноотрицательными срабатываниями и выбрать модель, подходящую под конкретную бизнес-задачу.

**2. Зачем для анализа качества модели делить датасет на обучающую и тренировочную выборки?**  
Оценка качества на тех же данных, на которых модель обучалась, даёт завышенный (оптимистичный) результат, так как модель могла просто запомнить ответы. Разделение позволяет проверить обобщающую способность – способность модели правильно предсказывать на новых, ранее не виденных данных. Это единственный способ оценить, как модель будет работать в реальных условиях.

**3. В чем особенность и область применения каждой метрики качества?**  
- **Accuracy (точность)**: доля правильных ответов. Хороша при сбалансированных классах.  
- **Precision (точность положительных предсказаний)**: из всех предсказанных положительных – сколько действительно положительных. Важна, когда ложноположительные срабатывания дороги (спам-фильтр: пометить хорошее письмо как спам).  
- **Recall (полнота)**: из всех реальных положительных – сколько модель нашла. Важна, когда ложноотрицательные срабатывания опасны (диагностика рака: пропустить болезнь).  
- **F1-мера**: среднее гармоническое precision и recall. Используется, когда нужен баланс между ними.  
- **Confusion matrix (матрица ошибок)**: показывает все четыре исхода (TP, TN, FP, FN) и позволяет детально разобрать ошибки по каждому классу.

**4. Для каждой метрики классификации и регрессии придумайте пример, в котором данная метрика будет определяющей для выбора наилучшей модели.**  
- **Accuracy**: задача угадывания пола по росту и весу (классы примерно поровну).  
- **Precision**: поисковая выдача – важно, чтобы на первой странице были только релевантные результаты.  
- **Recall**: детекция террористических угроз – лучше перестраховаться и проверить лишнего, чем пропустить реальную угрозу.  
- **F1**: чат-бот – важно и отвечать по делу (precision), и не пропускать вопросы (recall).  
- **MAE (регрессия)**: прогноз температуры – проще интерпретировать в градусах.  
- **MSE/RMSE**: штрафует большие ошибки сильнее – подходит, когда крупные отклонения недопустимы (управление ракетой).  
- **R²**: сравнение качества моделей на разных шкалах целевой переменной.

**5. В каких случаях нельзя делить выборку случайным образом?**  
- Данные имеют временную структуру (временные ряды, прогнозирование курса акций) – тогда нужно делить по времени: обучающая выборка – прошлое, тестовая – будущее, иначе произойдёт «подглядывание в будущее» (data leakage).  
- Данные сгруппированы (несколько образцов от одного пациента) – тогда нужно делить по группам, чтобы один пациент не попал и в обучение, и в тест.  
- Данные имеют пространственную автокорреляцию (например, спутниковые снимки соседних участков) – требуется специальное пространственное разделение.

**6. Зачем нужен и как использовать отчет о классификации?**  
Отчёт о классификации (classification_report) выдаёт для каждого класса значения precision, recall, f1-score и support (количество реальных примеров класса). Он позволяет быстро оценить, на каких классах модель работает хорошо, а на каких плохо, увидеть дисбаланс метрик и принять решение о необходимости сбора данных по плохо распознаваемому классу или выборе другой модели. Используется после обучения, на тестовой выборке, передавая в функцию истинные и предсказанные метки.